# Create Feature02.csv

產生金額相關的特徵，未經過匯率轉換

Output: `feature02.csv`  

Form:
| acct | send_sum | send_count | send_mean | send_std | send_min | send_max | recv_sum | recv_count | recv_mean | recv_std | recv_min | recv_max | 
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |

Descriptions:
| Column | Description | Range |
| --- | --- | --- |
| send_sum | 帳號「匯款」的金額總和 |  float |
| send_count | 帳號「匯款」的次數 | int |
| send_mean | 帳號「匯款」的金額平均 | float |
| send_std | 帳號「匯款」的金額標準差 | float |
| send_min | 帳號「匯款」的金額最小值 | float |
| send_max | 帳號「匯款」的金額最大值 | float |
| --- | --- | --- |
| recv_sum |  帳號「收款」的金額總和 | float |
| recv_count | 帳號「收款」的次數 | int |
| recv_mean | 帳號「收款」的金額平均 | float |
| recv_std | 帳號「收款」的金額最大值 | float |
| recv_min | 帳號「收款」的金額最小值 | float |
| recv_max | 帳號「收款」的金額最大值 | float |

In [ ]:
import pandas as pd

### Read files

In [ ]:
column_types = {
    "from_acct": "category",
    "from_acct_type": "category",
    "to_acct": "category",
    "to_acct_type": "category",
    "is_self_txn": "category",
    "txn_amt": "float",
    "txn_date": "category",
    "txn_time": "category",
    "currency_type": "category",
    "channel_type": "category"
}
alert_types = {
    "acct": "category",
    "event_date": "category"
}

df_txn = pd.read_csv("src/acct_transaction.csv", dtype=column_types)
df_alert = pd.read_csv("src/acct_alert.csv", dtype=alert_types)

print("CSV files read complete.")

### 處理資料

In [ ]:
# get send stats
print("getting send stats...")
send_stats = df_txn.groupby('from_acct').agg(
    send_sum=('txn_amt', 'sum'), 
    send_count=('txn_amt', 'count'), 
    send_std=('txn_amt', 'std'), 
    send_mean=('txn_amt', 'mean'), 
    send_min=('txn_amt', 'min'),
    send_max=('txn_amt', 'max')
).reset_index().rename(columns={'from_acct': 'acct'})

# get recv stats
print("getting recv stats...")
recv_stats = df_txn.groupby('to_acct').agg(
    recv_sum=('txn_amt', 'sum'), 
    recv_count=('txn_amt', 'count'), 
    recv_std=('txn_amt', 'std'), 
    recv_mean=('txn_amt', 'mean'), 
    recv_min=('txn_amt', 'min'), 
    recv_max=('txn_amt', 'max')
).reset_index().rename(columns={'to_acct': 'acct'})

# merge send and recv stats
print("merging send and recv stats...")
acct_stats = pd.merge(send_stats, recv_stats, on='acct', how='outer').fillna(0)

### 輸出

In [ ]:
acct_stats.to_csv("feature02.csv", index=False)